# Short strangles by IV rank

Chapter 3 of *The Unlucky Investor's Guide to Options Trading* asks when selling premium pays. This notebook runs the 16 delta strangle at 45 DTE (`options/premium/iv_ranges.py`), one backtest per IV rank range: it opens a strangle only while the rank of DVOL over the last year is inside the range. Then it compares the probability of profit and the P/L of the trades across the ranges.

Set `SBT2_SERVER_TOKEN`, and `SBT2_SERVER_URL` if the server is not on this machine. The server needs the Deribit data of the period and a year of DVOL before it.

In [ ]:
import os

import pandas as pd

from sbt2.lab import Lab

url = os.environ.get("SBT2_SERVER_URL", "ws://127.0.0.1:8765")
lab = await Lab.connect(url, os.environ["SBT2_SERVER_TOKEN"])

In [ ]:
spec = {
    "instruments": ["BTC-OPTIONS.DERIBIT"],
    "period": ["2024-01-01", "2025-01-01"],
    "execution": {
        "fee_model": {
            "kind": "capped_option",
            "config": {
                "maker_rate": "0.0003",
                "taker_rate": "0.0003",
                "cap_rate": "0.125",
            },
        }
    },
    "capital": "1 BTC",
    "split": {"validation": 0.2, "test": 0.2},
    "part": "train",
    "chain": {"min_dte": 0, "max_dte": 60, "strike_band": "0.3"},
}
ranges = [(0.0, 0.25), (0.25, 0.5), (0.5, 0.75), (0.75, 1.0)]

In [ ]:
by_range = {}
for low, high in ranges:
    runs = await lab.run(
        {**spec, "params": {"min_iv_rank": low, "max_iv_rank": high}},
        "options.premium.iv_ranges:ShortStrangle",
    )
    by_range[f"{low:.2f}-{high:.2f}"] = await runs[0].trade_statistics()
pd.DataFrame(by_range).T.rename_axis("IV rank")

In [ ]:
await lab.close()